---
title: Currency Exchange form API with Python
author: Luca Hindelang
date: 2026-05-10
toc: True
format:
    html:
        embed-resources: false

---

---

### Setup

- **Packages**

In [ ]:
!pip install requests
!pip install pandas
!pip install plotly.express
#| ouput: false

- **Imports**

In [2]:
from datetime import datetime, date, timedelta
import time
import requests
import pandas as pd
import plotly.express as px
import plotly.io as pio

- **Constants**

In [3]:
backdating_weeks = 52
base = "EUR"
quote = "CHF"
base_value = 100
quote_value = 100

pio.renderers.default = "notebook_connected"

---

### Fetching Data from API

- **Calculating Start Date**

In [4]:
start_date_list = str(datetime.now() - timedelta(weeks = backdating_weeks))
start_date = start_date_list.split(" ")[0]

- **API**

In [5]:
url_base_quote = f"https://api.frankfurter.dev/v2/rates?base={base}&from={start_date}&quotes={quote}"
url_quote_base = f"https://api.frankfurter.dev/v2/rates?base={quote}&from={start_date}&quotes={base}"

- **Overview Of Fetched Data**

The response from the API is a list of JSON-Objects.  
Applying the json() method on the response you can convert the list of JSON-Objects into a list of dictionaries.

In [6]:
response_base_quote = requests.get(url_base_quote)
response_quote_base = requests.get(url_quote_base)

data_base_quote = response_base_quote.json()
data_quote_base = response_quote_base.json()

print(data_base_quote[0])
print(data_quote_base[0])

{'date': '2025-05-11', 'base': 'EUR', 'quote': 'CHF', 'rate': 0.93505}
{'date': '2025-05-11', 'base': 'CHF', 'quote': 'EUR', 'rate': 1.0695}


---

### Adding Data To Pandas DataFrame

In [7]:
df_base_quote = pd.DataFrame(data_base_quote)
df_quote_base = pd.DataFrame(data_quote_base)

- **Overview of DataFrame**

In [8]:
df_base_quote.head()

,date,base,quote,rate
0,2025-05-11,EUR,CHF,0.93505
1,2025-05-11,EUR,CHF,0.93505
2,2025-05-12,EUR,CHF,0.93682
3,2025-05-13,EUR,CHF,0.93758
4,2025-05-14,EUR,CHF,0.93918


In [9]:
df_quote_base.head()

,date,base,quote,rate
0,2025-05-11,CHF,EUR,1.0695
1,2025-05-11,CHF,EUR,1.0695
2,2025-05-12,CHF,EUR,1.0674
3,2025-05-13,CHF,EUR,1.0666
4,2025-05-14,CHF,EUR,1.0648


---

### Plot Charts

- **Quote / Base Exchange Rate**

In [10]:
px.line(df_base_quote, 
        x = "date", y = "rate", 
        title = f"{quote}/{base} Exchange Rate", 
        labels = {"date": "Date", "rate": "Exchange Rate"}, 
        height = 600, width = 800
        )

- **Summary**

In [11]:
print(f"The current exchange rate is: {df_base_quote["rate"].iloc[-1]:.4f} {quote} per {base}")
print(f"Within the last {backdating_weeks} weeks, the exchange rate has been between {df_base_quote["rate"].min():.4f} and {df_base_quote["rate"].max():.4f}")
print(f"The average exchange rate within the last {backdating_weeks} weeks was {df_base_quote["rate"].mean():.4f}")
print(f"This means that {base_value} {base} is worth {quote_value * df_base_quote["rate"].iloc[-1]:.4f} {quote}")

The current exchange rate is: 0.9157 CHF per EUR
Within the last 52 weeks, the exchange rate has been between 0.9025 and 0.9434
The average exchange rate within the last 52 weeks was 0.9287
This means that 100 EUR is worth 91.5740 CHF


- **Base / Quote Exchange Rate**

In [12]:
px.line(df_quote_base,
        x = "date", y = "rate",
        title = f"{base}/{quote} Exchange Rate",
        labels = {"date": "Date", "rate": "Exchange Rate"},
        height = 600, width = 800
        )

- **Summary**

In [13]:
print(f"The current exchange rate is: {df_quote_base["rate"].iloc[-1]} {base} per {quote}")
print(f"Within the last {backdating_weeks} weeks, the exchange rate has been between {df_quote_base["rate"].min()} and {df_quote_base["rate"].max()}")
print(f"The average exchange rate within the last {backdating_weeks} weeks was {df_quote_base["rate"].mean():.3f}")
print(f"This means that {quote_value} {quote} is worth {base_value * df_quote_base["rate"].iloc[-1]:.4f} {base}")

The current exchange rate is: 1.092 EUR per CHF
Within the last 52 weeks, the exchange rate has been between 1.06 and 1.108
The average exchange rate within the last 52 weeks was 1.077
This means that 100 CHF is worth 109.2000 EUR


---


### Export

In [ ]:
!quarto render index.ipynb --to html
#| output: false

---

### Reference

- **[Quarto](https://quarto.org/docs/tools/vscode/index.html)**
- **[Frankfurter API](https://api.frankfurter.dev/)**